# HistoBreastNet — Preprocessing e costruzione del subset

Questo notebook prepara i dati per gli esperimenti di classificazione binaria (`benign` / `malignant`) su immagini istopatologiche del dataset **BreakHis_v1**. In particolare:

- analizza la struttura del dataset ed estrae i metadata dai path e dai nomi dei file;
- costruisce un **subset ridotto** (~1,5–1,6 GB), necessario perché il dataset completo è troppo pesante per il progetto, preservando il più possibile la distribuzione originale per classe, sottotipo e magnification;
- effettua la riduzione a **livello di paziente**, selezionando pazienti interi e non immagini singole;
- genera gli split sperimentali **image-wise**, **patient-wise** e **k-fold patient-wise**;
- salva i CSV e la copia fisica delle immagini del subset, che costituiscono l'input dei notebook successivi.

Il notebook **non allena alcun modello**: si limita a costruire un subset metodologicamente coerente e i relativi split, su cui verranno condotti gli esperimenti successivi.

La configurazione prodotta è `diversity_1p5GB` (budget ~1,5 GB, selezione *diversity-preferring* a livello di paziente), salvata sotto `data/processed/`. Il bundle include `metadata_subset.csv`, `subset_manifest.csv`, `image_wise_split.csv`, `patient_wise_split.csv`, `patient_wise_folds.csv` (k=5), `statistics.csv` e `config.json`; tutti i CSV riportano le colonne `relative_path` e `dataset_config`.

**Prerequisito:** `notebooks/dataset_reduction.py` deve essere la versione **v4** (verificato nella Sezione 2).

## Sezione 0 — Setup e path

Montaggio di Google Drive e definizione dei path del progetto. Il seed globale (`RANDOM_STATE`) fissa la sorgente di casualità e rende riproducibili sia la selezione del subset sia gli split.

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, shutil, time
drive.mount('/content/drive')

# Path del progetto su Drive: radice, cartella dei notebook e cartella di output.
# RANDOM_STATE fissa il seed per rendere riproducibili selezione del subset e split.
PROJECT_ROOT   = Path('/content/drive/MyDrive/HistoBreastNet')
SRC_DIR        = PROJECT_ROOT/'notebooks'
DATA_PROCESSED = PROJECT_ROOT/'data'/'processed'
RANDOM_STATE   = 42

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT non risolve: {PROJECT_ROOT} (controlla la scorciatoia)"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT OK:", PROJECT_ROOT)
print("data/processed :", DATA_PROCESSED)

Mounted at /content/drive
PROJECT_ROOT OK: /content/drive/MyDrive/HistoBreastNet
data/processed : /content/drive/MyDrive/HistoBreastNet/data/processed


## Sezione 1 — Dataset in locale (robusto alle disconnessioni)

Strategia: si lavora da uno **ZIP** su Drive. La prima cella:
1. se il dataset è già completo in locale → non fa nulla;
2. se lo **zip** esiste su Drive → lo estrae (veloce, 1–3 min);
3. se lo zip non esiste ancora → copia da `data/original/` in modo **resumable** (se il runtime si disconnette, **rilancia la cella**: riprende da dove era).

La seconda cella crea lo zip **una volta sola**, così le sessioni future partono in pochi minuti.

In [ ]:
import subprocess, zipfile, shutil, time
from pathlib import Path

# Il dataset viene portato in locale (/content) per accessi molto piu' rapidi
# rispetto a Drive; la sorgente di lettura resta comunque in sola lettura.
BREAKHIS_ORIG   = PROJECT_ROOT/'data'/'original'/'BreaKHis_v1'       # cartella su Drive
BREAKHIS_ZIP    = PROJECT_ROOT/'data'/'original'/'BreaKHis_v1.zip'   # zip su Drive (creato una volta)
BREAKHIS_LOCAL  = Path('/content/breakhis/BreaKHis_v1')
BREAKHIS_SOURCE = BREAKHIS_LOCAL

def n_local():
    return sum(1 for _ in BREAKHIS_LOCAL.rglob('*.png')) if BREAKHIS_LOCAL.exists() else 0

# Tre casi gestiti in ordine: (1) dataset gia' completo in locale -> nessuna azione;
# (2) zip presente su Drive -> estrazione veloce; (3) altrimenti copia resumable da
# data/original, rilanciabile senza perdite dopo una eventuale disconnessione.
n = n_local()
if n == 7909:
    print("Dataset già completo in locale.")
elif BREAKHIS_ZIP.exists():
    shutil.rmtree('/content/breakhis', ignore_errors=True)
    BREAKHIS_LOCAL.mkdir(parents=True, exist_ok=True)
    t0 = time.time(); print("Estrazione dallo zip su Drive...")
    with zipfile.ZipFile(BREAKHIS_ZIP) as z:
        z.extractall(BREAKHIS_LOCAL)
    n = n_local(); print(f"Estratto in {time.time()-t0:.0f}s")
else:
    BREAKHIS_LOCAL.mkdir(parents=True, exist_ok=True)
    print("Zip non presente: copio da data/original (resumable).")
    print("Se il runtime si disconnette, RILANCIA questa cella: riprende da dove era.")
    t0 = time.time()
    subprocess.run(['rsync','-a','--ignore-existing',
                    str(BREAKHIS_ORIG)+'/', str(BREAKHIS_LOCAL)+'/'])
    n = n_local(); print(f"{time.time()-t0:.0f}s")

print("Immagini .png:", n, "-> OK ✓" if n==7909 else f"-> [INCOMPLETO {n}] rilancia la cella")

# Controllo bloccante: senza le 7909 immagini attese il subset non sarebbe
# rappresentativo, quindi l'esecuzione si interrompe qui.
assert n == 7909, f"Dataset locale incompleto: trovate {n} immagini, attese 7909. Rilancia la cella prima di proseguire."

# Normalizza la root effettiva: a seconda di come e' strutturato lo zip,
# histology_slides/ puo' trovarsi direttamente sotto la cartella locale oppure
# dentro una sottocartella BreaKHis_v1/. BREAKHIS_SOURCE punta al livello corretto.
if (BREAKHIS_LOCAL / "histology_slides").exists():
    BREAKHIS_SOURCE = BREAKHIS_LOCAL
elif (BREAKHIS_LOCAL / "BreaKHis_v1" / "histology_slides").exists():
    BREAKHIS_SOURCE = BREAKHIS_LOCAL / "BreaKHis_v1"
else:
    raise FileNotFoundError(
        f"Non trovo histology_slides dentro {BREAKHIS_LOCAL}. "
        "Controlla la struttura dello zip estratto."
    )
print("BREAKHIS_SOURCE effettivo:", BREAKHIS_SOURCE)

Estrazione dallo zip su Drive...
Estratto in 102s
Immagini .png: 7909 -> OK ✓
BREAKHIS_SOURCE effettivo: /content/breakhis/BreaKHis_v1


In [ ]:
# UNA TANTUM: crea lo zip su Drive (dalla copia locale, veloce) per le sessioni future.
# Eseguila solo quando il conteggio sopra è 7909. Non fa nulla se lo zip esiste già.
if n_local() == 7909 and not BREAKHIS_ZIP.exists():
    t0 = time.time(); print("Creo lo zip su Drive dalla copia locale...")
    shutil.make_archive(str(BREAKHIS_ZIP.with_suffix('')), 'zip', BREAKHIS_LOCAL)
    print(f"Zip creato in {time.time()-t0:.0f}s ->", BREAKHIS_ZIP)
elif BREAKHIS_ZIP.exists():
    print("Zip già presente su Drive:", BREAKHIS_ZIP)
else:
    print("Copia locale non ancora completa (serve 7909): completa la cella precedente prima.")

Zip già presente su Drive: /content/drive/MyDrive/HistoBreastNet/data/original/BreaKHis_v1.zip


## Sezione 2 — Import del modulo di preprocessing

Le funzioni di estrazione dei metadata, costruzione del subset e generazione degli split sono raccolte nel modulo `dataset_reduction.py`. Il modulo viene importato con `importlib.reload`, così che nella sessione corrente venga sempre usata l'ultima versione del file.

In [ ]:
module_drive = SRC_DIR/'dataset_reduction.py'
assert module_drive.exists(), "dataset_reduction.py non è in notebooks/. Caricalo prima."

# reload esplicito: se il modulo viene aggiornato durante la sessione, garantisce
# che venga usata l'ultima versione e non quella eventualmente in cache.
sys.path.insert(0, str(SRC_DIR))
import importlib, dataset_reduction as dr; importlib.reload(dr)

<module 'dataset_reduction' from '/content/drive/MyDrive/HistoBreastNet/notebooks/dataset_reduction.py'>

## Sezione 3 — Estrazione dei metadata (config-independent)

Da ogni immagine del dataset completo vengono estratti i metadata a partire dalla struttura dei path e dai nomi dei file. La tabella descrive tutte le immagini (82 pazienti) ed è indipendente dalla configurazione del subset.

Colonne principali e loro ruolo:

- `path` / `relative_path`: percorso assoluto (locale) e percorso relativo a partire da `histology_slides/...`. Il `relative_path` è usato da tutti i CSV per ricostruire i percorsi in modo indipendente dalla sessione;
- `filename`: nome del file immagine;
- `label`: classe testuale (`benign` / `malignant`);
- `binary_label`: target binario numerico (0 = benign, 1 = malignant), usato dai modelli;
- `subtype` / `subtype_name`: sottotipo tumorale, mantenuto per le analisi degli errori nei notebook successivi;
- `magnification`: fattore di ingrandimento (40X, 100X, 200X, 400X), mantenuto per valutare la robustezza rispetto all'ingrandimento;
- `patient_id`: identificativo del paziente, indispensabile per costruire split privi di leakage.

Sottotipo e magnification non sono il target della classificazione, ma vengono conservati nei metadata perché necessari alle analisi successive.

In [ ]:
# extract_metadata ricava classe, sottotipo, magnification e patient_id analizzando
# i path e i nomi dei file BreakHis. La tabella copre l'intero dataset (attesi 82
# pazienti) ed e' indipendente dalla configurazione del subset.
metadata_full = dr.extract_metadata(BREAKHIS_SOURCE)
n = metadata_full['patient_id'].nunique()
metadata_full.to_csv(DATA_PROCESSED/'metadata_full.csv', index=False)
print("Immagini:", len(metadata_full), "| pazienti:", n, "-> OK 82 ✓" if n==82 else f"-> [nota] {n}")
print("Colonne (nota relative_path):", list(metadata_full.columns))

## Sezione 4 — Costruzione del subset `diversity_1p5GB` e degli split

### Riduzione a livello di paziente

La riduzione del dataset viene effettuata a **livello di paziente**: si selezionano pazienti interi mantenendo tutte le loro immagini, invece di campionare immagini singole. In questo modo si evita di costruire un subset artificiale in cui alcune immagini di uno stesso paziente sono incluse e altre escluse. Questa scelta è coerente con la valutazione patient-wise adottata nei notebook successivi e riduce il rischio di leakage tra train, validation e test, poiché il paziente è l'unità minima di campionamento.

### Strategia di selezione

La selezione è *diversity-preferring* con un budget di dimensione di circa 1,5 GB e cerca di **mantenere una distribuzione ragionevolmente simile** a quella del dataset originale rispetto a classe (`benign` / `malignant`), sottotipo tumorale, magnification (40X, 100X, 200X, 400X) e varietà di pazienti. Non si impone una corrispondenza esatta delle proporzioni, ma si preserva il più possibile la struttura del dataset di partenza.

### Split sperimentali

Sullo stesso subset vengono generati tre tipi di split:

- **image-wise**: le immagini sono suddivise senza vincoli sul paziente. Utile come riferimento iniziale, ma potenzialmente **ottimistico**, perché immagini dello stesso paziente possono comparire sia in train sia in test;
- **patient-wise**: train, validation e test contengono pazienti **disgiunti**. Fornisce una stima più realistica della capacità di generalizzazione a pazienti non visti;
- **k-fold patient-wise** (k=5): il subset è diviso in più fold paziente-disgiunti e ogni paziente compare nel test di un solo fold. È il **protocollo principale** del progetto, perché valuta il modello su più partizioni indipendenti e produce una stima più stabile rispetto a un singolo split.

La valutazione principale nei notebook successivi si basa sul **k-fold patient-wise**.

In [ ]:
# Selezione del subset a livello di paziente (unita' minima di campionamento): tutte
# le immagini di un paziente scelto vengono mantenute insieme. Budget ~1.5 GB e
# strategia diversity-preferring preservano il piu' possibile la distribuzione
# originale per classe, sottotipo e magnification.
sel_b, sub_b, man_b = dr.build_subset_budget(metadata_full, target_gb=1.5, random_state=RANDOM_STATE)
# Split image-wise (riferimento, potenzialmente ottimistico) e patient-wise
# (pazienti disgiunti tra train/validation/test, stima piu' realistica).
iw_b, pw_b = dr.make_splits(sub_b, random_state=RANDOM_STATE)
# k-fold patient-wise (k=5): protocollo principale del progetto. Ogni fold ha
# pazienti disgiunti e ogni paziente compare nel test una sola volta.
kf_b = dr.make_kfold_patient_splits(sub_b, k=5, random_state=RANDOM_STATE)

# Salvataggio del bundle completo (CSV + config.json) della configurazione.
dir_b, cfg_b = dr.write_config_bundle(
    sub_b, man_b, iw_b, pw_b, kf_b, DATA_PROCESSED/'diversity_1p5GB',
    dataset_config='diversity_1p5GB', selection_strategy='diversity_preferring_patient_level',
    random_state=RANDOM_STATE, target_size_gb=1.5)

print("Config B ->", dir_b)
print(f"  pazienti={cfg_b['n_patients']} immagini={cfg_b['n_images']} "
      f"size={cfg_b['actual_size_gib']} GiB ({cfg_b['actual_size_gb_decimal']} GB dec)")
print("  per sottotipo:", cfg_b['subtypes'])
print("  classe (pazienti):", cfg_b['n_patients_benign'], "benign /", cfg_b['n_patients_malignant'], "malignant")
print("  test patient-wise (n pazienti):", pw_b[pw_b.split=='test']['patient_id'].nunique())
print("  fold validi:", cfg_b['folds_valid'])

## Sezione 5 — Verifica finale (file prodotti e controllo anti-leakage sui fold)

Per ogni configurazione presente in `data/processed/` vengono elencati i file del bundle e viene verificata la validità dei fold patient-wise. Il controllo garantisce che nessun paziente compaia contemporaneamente in train, validation e test e che ogni paziente sia usato come test in un solo fold, requisito fondamentale per una valutazione priva di leakage.

In [ ]:
import pandas as pd
# Per ogni configurazione prodotta elenca i file del bundle e verifica i fold:
# verify_folds controlla l'assenza di sovrapposizione di pazienti tra i fold e
# che ogni paziente sia usato come test esattamente una volta.
for cfg_dir in sorted(DATA_PROCESSED.glob('*/')):
    if not (cfg_dir/'config.json').exists():
        continue
    print(f"== {cfg_dir.name} ==")
    for f in sorted(cfg_dir.glob('*')):
        print(f"   • {f.name:26s} {f.stat().st_size/1024:8.1f} KB")
    kf = pd.read_csv(cfg_dir/'patient_wise_folds.csv')
    print("   fold validi (zero overlap + test una volta):", dr.verify_folds(kf, k=kf['fold'].nunique()), "\n")

## Sezione 6 — Copia fisica del subset (`images/`)

Crea una copia fisica delle **sole** immagini del subset `diversity_1p5GB` sotto `data/processed/diversity_1p5GB/images/`, mantenendo la struttura relativa che parte da `histology_slides/...`. Serve a rendere il training più veloce (evita la lettura di migliaia di file sparsi da `data/original/`) e permette ai notebook successivi di ricostruire i percorsi come `DATASET_ROOT / relative_path`, con `DATASET_ROOT = .../images`.

- **Non modifica mai** `data/original/BreaKHis_v1`: accesso in sola lettura, senza spostamenti, cancellazioni o rinomine.
- **Idempotente e resumable**: se il file di destinazione esiste con la **stessa** dimensione non viene ricopiato; se ha dimensione **diversa** viene sovrascritto. In caso di disconnessione è sufficiente rilanciare la cella, che riprende da dove si era interrotta.
- Gli split restano gestiti dai CSV: **nessuna** sottocartella `train/`, `val/` o `test/`.
- `relative_path` resta invariata; viene solo, opzionalmente, **aggiunta** la colonna `subset_relative_path` (identica a `relative_path`).

In [ ]:
import pandas as pd
from pathlib import Path
import shutil, time

# ---------------- Parametri della copia fisica ----------------
DATASET_CONFIG         = "diversity_1p5GB"
CREATE_PHYSICAL_SUBSET = True          # copia fisica del subset
ADD_SUBSET_RELPATH_COL = True          # aggiunge 'subset_relative_path' (== relative_path), senza sostituirla

PROCESSED_DIR          = DATA_PROCESSED / DATASET_CONFIG
METADATA_SUBSET_CSV    = PROCESSED_DIR / 'metadata_subset.csv'
SUBSET_IMAGES_DIR      = PROCESSED_DIR / 'images'
ORIGINAL_DATASET_ROOT  = PROJECT_ROOT / 'data' / 'original' / 'BreaKHis_v1'   # MAI modificato

# Sorgente di LETTURA: preferisci la copia locale già estratta in Sezione 1 (molto più veloce),
# altrimenti fallback sull'originale su Drive. In entrambi i casi è sola lettura.
try:
    _local_root = BREAKHIS_SOURCE                      # definito in Sezione 1 se disponibile
except NameError:
    _local_root = Path('/content/breakhis/BreaKHis_v1')
SOURCE_ROOT = _local_root if _local_root.exists() else ORIGINAL_DATASET_ROOT
assert SOURCE_ROOT.exists(), f"Nessuna sorgente valida: {_local_root} oppure {ORIGINAL_DATASET_ROOT}"

print("Sorgente lettura :", SOURCE_ROOT, "(locale)" if SOURCE_ROOT == _local_root else "(Drive)")
print("Destinazione     :", SUBSET_IMAGES_DIR)

if not CREATE_PHYSICAL_SUBSET:
    print("CREATE_PHYSICAL_SUBSET=False -> salto la copia.")
else:
    assert METADATA_SUBSET_CSV.exists(), f"Manca {METADATA_SUBSET_CSV} (esegui la Sezione 5 prima)."
    df = pd.read_csv(METADATA_SUBSET_CSV)
    assert 'relative_path' in df.columns, "Colonna 'relative_path' assente in metadata_subset.csv"

    SUBSET_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    n_csv = len(df)
    n_present = n_copied = n_missing = 0
    missing = []
    t0 = time.time()
    for i, rel in enumerate(df['relative_path'].astype(str), 1):
        src = SOURCE_ROOT / rel
        if not src.exists() and SOURCE_ROOT != ORIGINAL_DATASET_ROOT:
            src = ORIGINAL_DATASET_ROOT / rel      # fallback: copia locale incompleta -> leggi da Drive
        dst = SUBSET_IMAGES_DIR / rel
        if not src.exists():
            n_missing += 1; missing.append(rel); continue
        s_size = src.stat().st_size
        if dst.exists() and dst.stat().st_size == s_size:
            n_present += 1                              # già copiato, stessa dimensione -> skip
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)   # crea le sottocartelle necessarie
            shutil.copy2(src, dst)                     # copia (sovrascrive se dimensione diversa)
            n_copied += 1
        if i % 500 == 0:
            print(f"  ...{i}/{n_csv}  (presenti={n_present} copiate={n_copied} mancanti={n_missing})")
    dt = time.time() - t0

    # Colonna opzionale: NON sostituisce relative_path
    if ADD_SUBSET_RELPATH_COL:
        if 'subset_relative_path' not in df.columns:
            df['subset_relative_path'] = df['relative_path']
            df.to_csv(METADATA_SUBSET_CSV, index=False)
            print("Aggiunta colonna 'subset_relative_path' a metadata_subset.csv")
        else:
            print("Colonna 'subset_relative_path' già presente: nessuna modifica.")

    total_bytes = sum(p.stat().st_size for p in SUBSET_IMAGES_DIR.rglob('*.png'))
    print("\n===== RIEPILOGO COPIA =====")
    print(f"immagini nel CSV     : {n_csv}")
    print(f"gia' presenti (skip) : {n_present}")
    print(f"copiate ora          : {n_copied}")
    print(f"mancanti (sorgente)  : {n_missing}")
    print(f"dimensione images/   : {total_bytes/1024**3:.3f} GiB ({total_bytes/1e9:.3f} GB dec)")
    print(f"tempo totale         : {dt:.0f}s")
    if missing:
        print("Esempi mancanti:", missing[:10])

Sorgente lettura : /content/breakhis/BreaKHis_v1 (locale)
Destinazione     : /content/drive/MyDrive/HistoBreastNet/data/processed/diversity_1p5GB/images
  ...500/2838  (presenti=0 copiate=500 mancanti=0)
  ...1000/2838  (presenti=0 copiate=1000 mancanti=0)
  ...1500/2838  (presenti=0 copiate=1500 mancanti=0)
  ...2000/2838  (presenti=0 copiate=2000 mancanti=0)
  ...2500/2838  (presenti=0 copiate=2500 mancanti=0)
Aggiunta colonna 'subset_relative_path' a metadata_subset.csv

===== RIEPILOGO COPIA =====
immagini nel CSV     : 2838
gia' presenti (skip) : 0
copiate ora          : 2838
mancanti (sorgente)  : 0
dimensione images/   : 1.483 GiB (1.592 GB dec)
tempo totale         : 76s


### Validazioni della copia fisica

Controlli bloccanti sulla copia appena prodotta:

- il numero di immagini `.png` presenti sotto `images/` coincide con il numero di righe di `metadata_subset.csv`;
- ogni `relative_path` del CSV esiste fisicamente sotto `images/` (nessun file mancante);
- le distribuzioni per `label`, `binary_label`, `subtype`, `magnification` e `patient_id` restano identiche a quelle di `metadata_subset.csv`;
- `relative_path` resta invariata e, se presente, `subset_relative_path` coincide con essa.

In [ ]:
import pandas as pd
df = pd.read_csv(METADATA_SUBSET_CSV)
rels = df['relative_path'].astype(str)

# 1) Il numero di .png sotto images/ deve coincidere con le righe del CSV:
#    nessuna immagine persa o in eccesso.
n_png_folder = sum(1 for _ in SUBSET_IMAGES_DIR.rglob('*.png'))
assert n_png_folder == len(df), f"Conteggio diverso: folder={n_png_folder} csv={len(df)}"

# 2) ogni relative_path del CSV esiste sotto images/ (zero mancanti)
missing_dst = [r for r in rels if not (SUBSET_IMAGES_DIR / r).exists()]
assert not missing_dst, f"Mancano {len(missing_dst)} file sotto images/, es: {missing_dst[:5]}"

# 3) Le distribuzioni per classe, sottotipo, magnification e paziente devono
#    restare identiche: la copia non altera la composizione del subset.
present = df[df['relative_path'].map(lambda r: (SUBSET_IMAGES_DIR / r).exists())]
for col in ['label', 'binary_label', 'subtype', 'magnification', 'patient_id']:
    a = df[col].value_counts().sort_index()
    b = present[col].value_counts().sort_index()
    assert a.equals(b), f"Distribuzione '{col}' cambiata"

# 4) relative_path resta invariata; subset_relative_path (se presente) è identica
assert 'relative_path' in df.columns, "relative_path mancante!"
if 'subset_relative_path' in df.columns:
    assert (df['subset_relative_path'].astype(str) == rels).all(), "subset_relative_path != relative_path"

print("Validazioni OK",
      "| immagini:", n_png_folder,
      "| pazienti:", df['patient_id'].nunique(),
      "| subtypes:", df['subtype'].nunique(),
      "| magnifications:", df['magnification'].nunique())

Validazioni OK | immagini: 2838 | pazienti: 33 | subtypes: 8 | magnifications: 4


### Zip opzionale del subset

Crea `data/processed/diversity_1p5GB/images.zip` con struttura `images/histology_slides/...`, così da poter essere estratto in `/content/` e usato con `DATASET_ROOT = <estratto>/images`.

L'operazione è controllata dal flag `CREATE_IMAGES_ZIP`. Le cartelle `data/processed/*/images/` e i file `data/processed/*/images.zip` non vanno inclusi nel controllo di versione (vanno aggiunti a `.gitignore`).

In [ ]:
# Zip opzionale del subset: consente un'estrazione rapida in una nuova sessione.
CREATE_IMAGES_ZIP = True
SUBSET_IMAGES_ZIP = PROCESSED_DIR / 'images.zip'

if CREATE_IMAGES_ZIP:
    import shutil, time
    t0 = time.time()
    print("Creo/aggiorno lo zip del subset...")
    # root_dir=PROCESSED_DIR + base_dir='images' -> gli entry hanno prefisso 'images/...'
    shutil.make_archive(str(SUBSET_IMAGES_ZIP.with_suffix('')), 'zip',
                        root_dir=str(PROCESSED_DIR), base_dir='images')
    print(f"Zip creato in {time.time()-t0:.0f}s ->", SUBSET_IMAGES_ZIP,
          f"({SUBSET_IMAGES_ZIP.stat().st_size/1024**3:.3f} GiB)")
else:
    print("CREATE_IMAGES_ZIP=False -> nessuno zip creato.")

Creo/aggiorno lo zip del subset...
Zip creato in 120s -> /content/drive/MyDrive/HistoBreastNet/data/processed/diversity_1p5GB/images.zip (1.483 GiB)


## Output prodotti

Al termine dell'esecuzione la configurazione `diversity_1p5GB` e' disponibile sotto `data/processed/diversity_1p5GB/`:

- `metadata_subset.csv`: metadata completo del subset (una riga per immagine);
- `image_wise_split.csv`: split image-wise, usato come riferimento iniziale;
- `patient_wise_split.csv`: split train/validation/test con pazienti disgiunti;
- `patient_wise_folds.csv`: fold patient-wise (k=5) usati per la valutazione principale;
- `subset_manifest.csv`: riepilogo per paziente (numero di immagini e dimensione);
- `statistics.csv`: distribuzioni per classe, sottotipo e magnification;
- `config.json`: parametri della configurazione (strategia, seed, dimensioni, conteggi);
- `images/`: copia fisica delle immagini del subset (con l'eventuale `images.zip`).

Questi file costituiscono l'input degli esperimenti successivi: i modelli vengono addestrati e valutati a partire dagli split qui definiti, adottando il k-fold patient-wise come protocollo principale.